# QDFL corrected figure regeneration (Figs 7-11)

This notebook regenerates the data-driven figures of the QDFL manuscript from the
**executed leakage-safe** notebook outputs (seed 42, real data, zero errors). It loads
only authoritative corrected values and never the old leaky pipeline or old embedded figures.

## Provenance

| Figure | Quantity | Source (executed leakage-safe) | Matching table |
|---|---|---|---|
| Fig 7 | Single-split ROC-AUC, 14 models | qdfl-hybrid-leakage-safe.ipynb | Table IV |
| Fig 8 | Uhlmann fidelity vs noise p | qdfl-hybrid-with-noise-auc-leakage-safe.ipynb | Table VI |
| Fig 9 | ROC-AUC vs trainable params | qdfl-hybrid-leakage-safe.ipynb | Table IV |
| Fig 10 | Cross-dataset rank stability (rho) | qdfl-hybrid-leakage-safe.ipynb | Table IV + 4.3.3 |
| Fig 11 | Downstream QDFL ROC-AUC under noise | qdfl-hybrid-with-noise-auc-leakage-safe.ipynb | 4.4 |

**Figs 12 (confusion matrices) and 13 (ROC curves) are NOT regenerated here**: no executed
leakage-safe notebook emits per-sample predictions or the VQDVF model, so they are blocked
pending a corrected predictions export. They are not fabricated.

The values below are transcribed from the executed notebook outputs; every figure prints a
verification line so the plotted numbers can be checked against the manuscript tables.


In [1]:
import os
os.makedirs("figs", exist_ok=True)
# QDFL corrected figure regeneration — Figs 7-11 from executed leakage-safe outputs.
# DATA PROVENANCE (all values read from executed leakage-safe notebook outputs, seed 42):
#   Table IV single-split ROC/F1/AP + params  <- qdfl-hybrid-leakage-safe.ipynb
#   Table VI fidelity-by-p (VQC/QDCN/QDFL)     <- qdfl-hybrid-with-noise-auc-leakage-safe.ipynb
#   Rank table + Spearman rho                  <- qdfl-hybrid-leakage-safe.ipynb
#   Fig-11 downstream ROC-AUC under noise       <- qdfl-hybrid-with-noise-auc-leakage-safe.ipynb (noise_auc.csv)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'font.size':11,'font.family':'DejaVu Sans','axes.grid':True,'grid.alpha':0.3})

# ---------- Table IV single-split (14 models) ----------
# name: (params, primaryROC, primF1, primAP, paysimROC, paysF1, paysAP, family)
M = {
 'XGBoost':            (26444,0.8888,0.8848,0.8399,0.9991,0.9950,0.9991,'classical'),
 'Random Forest':      (67336,0.8864,0.8903,0.8361,0.9989,0.9919,0.9993,'classical'),
 'Neural Network':     (10529,0.8880,0.8901,0.8317,0.9970,0.9756,0.9962,'classical'),
 'Logistic Regression':(17,   0.8730,0.8786,0.7565,0.9904,0.9528,0.9917,'classical'),
 'VQC':                (137,  0.8816,0.8785,0.8506,0.9330,0.8750,0.9397,'quantum'),
 'QDCN (standalone)':  (4417, 0.8754,0.8462,0.8178,0.9362,0.8738,0.9407,'quantum'),
 'QSVM':               (0,    0.8628,0.8304,0.7359,0.9402,0.8638,0.9515,'quantum'),
 'VQC-matched (ablation)':(65,0.4998,0.5569,0.5923,0.9106,0.8459,0.9211,'quantum'),
 'FedProx-NONIID':     (23905,0.8895,0.8908,0.8283,0.9912,0.9526,0.9912,'federated'),
 'FedAvg-IID':         (23905,0.8862,0.8906,0.8274,0.9924,0.9618,0.9949,'federated'),
 'FedProx-IID':        (23905,0.8875,0.8898,0.8268,0.9920,0.9490,0.9938,'federated'),
 'FedAvg-NONIID':      (23905,0.8880,0.8906,0.8283,0.9941,0.9563,0.9925,'federated'),
 'QDFL-Hybrid-NONIID': (4937, 0.8860,0.8793,0.8263,0.9651,0.8780,0.9498,'qdfl'),
 'QDFL-Hybrid-IID':    (4937, 0.8733,0.8556,0.8185,0.9503,0.8972,0.9590,'qdfl'),
}
FAM_C={'classical':'#7f7f7f','quantum':'#1f77b4','federated':'#2ca02c','qdfl':'#d62728'}

In [2]:
# ================= FIG 7 : ranked ROC-AUC bars =================
def fig7():
    fig,ax=plt.subplots(1,2,figsize=(15,7))
    for k,(col,ttl) in enumerate([(1,'Primary (Azamuke 2024)'),(4,'PaySim (Lopez-Rojas)')]):
        data=sorted(M.items(),key=lambda kv:kv[1][col])   # ascending -> top is largest
        names=[n for n,_ in data]; vals=[v[col] for _,v in data]
        cols=['#d62728' if v[7]=='qdfl' else '#8a8a8a' for _,v in data]
        ax[k].barh(range(len(names)),vals,color=cols,edgecolor='none')
        ax[k].set_yticks(range(len(names))); ax[k].set_yticklabels(names,fontsize=9)
        for i,v in enumerate(vals): ax[k].text(v+ (0.002 if col==1 else 0.0008),i,f'{v:.4f}',va='center',fontsize=8)
        ax[k].set_xlabel('ROC-AUC (single seed = 42)')
        ax[k].set_title(f'{ttl}\n(red = QDFL variants)')
        ax[k].set_xlim( (0.45,1.0) if col==1 else (0.895,1.001) )
    plt.tight_layout(); plt.savefig('figs/fig7.png',dpi=200,bbox_inches='tight'); plt.savefig('figs/fig7.pdf',bbox_inches='tight'); plt.close()

# ================= FIG 8 : fidelity vs p =================
P=[0.001,0.005,0.01,0.05,0.1]
FID={ # dataset -> model -> [fidelity at each p]   (Table VI)
 'Primary':{'VQC':[0.9818,0.9121,0.8322,0.4043,0.1713],'QDCN':[0.9944,0.9725,0.9459,0.7588,0.5805],'QDFL':[0.9909,0.9552,0.9125,0.6379,0.4200]},
 'PaySim': {'VQC':[0.9817,0.9119,0.8317,0.4034,0.1709],'QDCN':[0.9941,0.9707,0.9422,0.7434,0.5560],'QDFL':[0.9906,0.9540,0.9102,0.6282,0.4047]},
}
def fig8():
    fig,ax=plt.subplots(1,2,figsize=(15,5.5))
    for k,ds in enumerate(['PaySim','Primary']):
        ttl='PaySim (Lopez-Rojas)' if ds=='PaySim' else 'Primary (Azamuke 2024)'
        ax[k].plot(P,FID[ds]['VQC'],'o-',color='#1f77b4',label='VQC')
        ax[k].plot(P,FID[ds]['QDCN'],'s-',color='#2ca02c',label='QDCN-standalone')
        ax[k].plot(P,FID[ds]['QDFL'],'^-',color='#d62728',label='QDFL-Hybrid')
        ax[k].set_xscale('log'); ax[k].set_xlabel('Noise probability p (log scale)')
        ax[k].set_ylabel('Mean Uhlmann fidelity'); ax[k].set_ylim(0.15,1.02)
        ax[k].set_title(f'{ttl}\nNoise-resilience curves (mean across channels \u00d7 50 inputs, seed 42)',fontsize=10)
        ax[k].legend(loc='upper right')
    plt.tight_layout(); plt.savefig('figs/fig8.png',dpi=200,bbox_inches='tight'); plt.savefig('figs/fig8.pdf',bbox_inches='tight'); plt.close()

# ================= FIG 9 : ROC-AUC vs params =================
def fig9():
    fig,ax=plt.subplots(1,2,figsize=(15,5.5))
    for k,(col,ttl) in enumerate([(4,'PaySim (Lopez-Rojas)'),(1,'Primary (Azamuke 2024)')]):
        for n,v in M.items():
            if v[0]==0: continue  # QSVM omitted (no trainable params)
            ax[k].scatter(v[0],v[col],s=90,color=FAM_C[v[7]],edgecolor='k',linewidth=0.5,zorder=3)
            ax[k].annotate(n,(v[0],v[col]),fontsize=7,xytext=(4,2),textcoords='offset points')
        ax[k].set_xscale('log'); ax[k].set_xlabel('Trainable parameter count (log scale)')
        ax[k].set_ylabel('ROC-AUC'); ax[k].set_title(f'{ttl}\nParameter-efficient Pareto (red = QDFL)')
    plt.tight_layout(); plt.savefig('figs/fig9.png',dpi=200,bbox_inches='tight'); plt.savefig('figs/fig9.pdf',bbox_inches='tight'); plt.close()

# ================= FIG 10 : rank stability =================
# authoritative ranks (qdfl-hybrid-leakage-safe.ipynb)
RANK_P={'FedProx-NONIID':1,'XGBoost':2,'Neural Network':3,'FedAvg-NONIID':4,'FedProx-IID':5,'Random Forest':6,'FedAvg-IID':7,'QDFL-Hybrid-NONIID':8,'VQC':9,'QDCN (standalone)':10,'QDFL-Hybrid-IID':11,'Logistic Regression':12,'QSVM':13,'VQC-matched (ablation)':14}
RANK_S={'XGBoost':1,'Random Forest':2,'Neural Network':3,'FedAvg-NONIID':4,'FedAvg-IID':5,'FedProx-IID':6,'FedProx-NONIID':7,'Logistic Regression':8,'QDFL-Hybrid-NONIID':9,'QDFL-Hybrid-IID':10,'QSVM':11,'QDCN (standalone)':12,'VQC':13,'VQC-matched (ablation)':14}
RHO={'ROC-AUC':(0.7802,0.0010),'F1-Score':(0.7026,0.0051),'Avg-Prec':(0.8901,0.00005)}
def fig10():
    fig,ax=plt.subplots(1,2,figsize=(16,7))
    # LEFT: score agreement scatter
    for n,v in M.items():
        red = v[7]=='qdfl'; c='#d62728' if red else '#8a8a8a'
        ax[0].scatter(v[1],v[4],marker='o',s=110 if red else 70,color=c,edgecolor='k',linewidth=0.6,zorder=3)  # ROC
        ax[0].scatter(v[2],v[5],marker='s',s=110 if red else 70,color=c,edgecolor='k',linewidth=0.6,zorder=3)  # F1
        ax[0].scatter(v[3],v[6],marker='^',s=110 if red else 70,color=c,edgecolor='k',linewidth=0.6,zorder=3)  # AP
    ax[0].plot([0.55,1.0],[0.55,1.0],'--',color='gray',label='y = x (perfect agreement)')
    from matplotlib.lines import Line2D
    leg=[Line2D([0],[0],marker='o',color='w',markerfacecolor='gray',markeredgecolor='k',markersize=9,label=f'ROC-AUC  (\u03c1 = +0.78, p = 0.001)'),
         Line2D([0],[0],marker='s',color='w',markerfacecolor='gray',markeredgecolor='k',markersize=9,label=f'F1-Score  (\u03c1 = +0.70, p = 0.005)'),
         Line2D([0],[0],marker='^',color='w',markerfacecolor='gray',markeredgecolor='k',markersize=9,label=f'Avg-Prec  (\u03c1 = +0.89, p < 0.001)'),
         Line2D([0],[0],ls='--',color='gray',label='y = x (perfect agreement)')]
    ax[0].legend(handles=leg,loc='lower right',fontsize=9)
    ax[0].set_xlabel('Score on Primary (Azamuke 2024)'); ax[0].set_ylabel('Score on PaySim (Lopez-Rojas)')
    ax[0].set_title('Cross-dataset score agreement\n(red = QDFL; o ROC-AUC, s F1, ^ Avg-Prec)')
    ax[0].set_xlim(0.54,1.01); ax[0].set_ylim(0.58,1.01)
    # RIGHT: rank movement slope chart
    for n in RANK_P:
        rp,rs=RANK_P[n],RANK_S[n]; red='QDFL' in n
        ax[1].plot([0,1],[rp,rs],'-',color='#d62728' if red else '#c8c8c8',lw=2.4 if red else 1.2,zorder=3 if red else 1)
        ax[1].scatter([0,1],[rp,rs],color='#d62728' if red else '#9a9a9a',s=45 if red else 30,zorder=4)
        ax[1].text(-0.02,rp,n,ha='right',va='center',fontsize=8,color='#d62728' if red else 'black',fontweight='bold' if red else 'normal')
        ax[1].text(1.02,rs,n,ha='left',va='center',fontsize=8,color='#d62728' if red else 'black',fontweight='bold' if red else 'normal')
    ax[1].set_ylim(14.6,0.4); ax[1].set_xlim(-0.55,1.55); ax[1].set_xticks([0,1]); ax[1].set_xticklabels(['Primary (Azamuke 2024)','PaySim (Lopez-Rojas)'])
    ax[1].set_ylabel('ROC-AUC rank (1 = best)'); ax[1].grid(False)
    ax[1].set_title('ROC-AUC rank movement across datasets\n(Spearman \u03c1 = +0.78, p = 0.001; flat line = stable)')
    plt.tight_layout(); plt.savefig('figs/fig10.png',dpi=200,bbox_inches='tight'); plt.savefig('figs/fig10.pdf',bbox_inches='tight'); plt.close()

# ================= FIG 11 : downstream noise-AUC bars =================
NOISE={ # dataset -> (free, {channel:(p01,p1)})
 'Azamuke 2024 (primary)':(0.8733,{'Bit flip':(0.8733,0.8739),'Phase flip':(0.8735,0.8675),'Amp. damping':(0.8735,0.8745),'Depolarizing':(0.8733,0.8734)}),
 'PaySim (comparison)':(0.9503,{'Bit flip':(0.9503,0.9504),'Phase flip':(0.9502,0.9479),'Amp. damping':(0.9502,0.9479),'Depolarizing':(0.9502,0.9497)}),
}
def fig11():
    fig,ax=plt.subplots(1,2,figsize=(15,6))
    for k,(ds,(free,ch)) in enumerate(NOISE.items()):
        chans=list(ch); x=np.arange(len(chans)); w=0.38
        p01=[ch[c][0] for c in chans]; p1=[ch[c][1] for c in chans]
        ax[k].bar(x-w/2,p01,w,color='#a6cee3',label='p = 0.01')
        ax[k].bar(x+w/2,p1,w,color='#1f5fa6',label='p = 0.1')
        ax[k].axhline(free,ls='--',color='#c0392b',label=f'noise-free ({free:.3f})')
        ax[k].set_xticks(x); ax[k].set_xticklabels(chans,rotation=15); ax[k].set_ylabel('ROC-AUC')
        ax[k].set_title(ds)
        lo=min(min(p01),min(p1),free); hi=max(max(p01),max(p1),free)
        ax[k].set_ylim(lo-0.004,hi+0.004); ax[k].legend(loc='lower left',fontsize=9)
    plt.tight_layout(); plt.savefig('figs/fig11.png',dpi=200,bbox_inches='tight'); plt.savefig('figs/fig11.pdf',bbox_inches='tight'); plt.close()

In [3]:
fig7(); fig8(); fig9(); fig10(); fig11()
print("figures written:")
import os
for f in sorted(os.listdir('figs')):
    if f.endswith('.png'): print("  ",f, os.path.getsize(f'figs/{f}'),"bytes")

figures written:
   fig10.png 383010 bytes
   fig11.png 134256 bytes
   fig7.png 227159 bytes
   fig8.png 208393 bytes
   fig9.png 175892 bytes


In [4]:
# ---- verification: plotted values vs manuscript tables ----
print("Fig 7  FedProx-NONIID primary ROC =", M['FedProx-NONIID'][1], "(Table IV: 0.8895)")
print("Fig 7  VQC primary ROC            =", M['VQC'][1], "(Table IV: 0.8816)")
print("Fig 8  QDCN primary fidelity p=.1 =", FID['Primary']['QDCN'][-1], "(Table VI: 0.5805)")
print("Fig 10 Spearman rho ROC-AUC       =", RHO['ROC-AUC'][0], "(4.3.3: +0.7802)")
print("Fig 10 FedProx-NONIID primary rank=", RANK_P['FedProx-NONIID'], "| PaySim rank =", RANK_S['FedProx-NONIID'])
print("Fig 11 QDFL noise-free primary AUC=", NOISE['Azamuke 2024 (primary)'][0], "| PaySim =", NOISE['PaySim (comparison)'][0])


Fig 7  FedProx-NONIID primary ROC = 0.8895 (Table IV: 0.8895)
Fig 7  VQC primary ROC            = 0.8816 (Table IV: 0.8816)
Fig 8  QDCN primary fidelity p=.1 = 0.5805 (Table VI: 0.5805)
Fig 10 Spearman rho ROC-AUC       = 0.7802 (4.3.3: +0.7802)
Fig 10 FedProx-NONIID primary rank= 1 | PaySim rank = 7
Fig 11 QDFL noise-free primary AUC= 0.8733 | PaySim = 0.9503


## Blocked figures

**Fig 12 (confusion matrices)** and **Fig 13 (ROC curves + VQDVF)** require per-sample
corrected predictions. None of the four executed leakage-safe notebooks computes confusion
matrices, ROC curves, or the VQDVF fusion model, and the only prediction files on disk are the
pre-correction (leaky) exports. To regenerate these, re-run the leakage-safe hybrid pipeline
with prediction saving enabled (and a VQDVF eval for Fig 13), export per-sample scores, then
plot. Until then these two figures must not be presented as corrected.
